# 03 - Treinamento EfficientNet-B2

Fine-tuning da EfficientNet-B2 pré-treinada no ImageNet para classificação binária de rostos reais vs gerados por IA.

1. Carregamento dos dados
2. Definição do modelo
3. Treinamento com early stopping
4. Curvas de loss e acurácia
5. Avaliação no conjunto de teste
6. Matriz de confusão e curva ROC

In [ ]:
import copy
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent
_data_root_file = PROJECT_ROOT / "data_root.env"
DATA_ROOT    = Path(_data_root_file.read_text().strip()) if _data_root_file.exists() else PROJECT_ROOT / "data"
RAW_DIR      = DATA_ROOT / "raw" / "140k_faces" / "real_vs_fake" / "real-vs-fake"
MODEL_DIR    = PROJECT_ROOT / "artifacts" / "models"
FIGURES_DIR  = PROJECT_ROOT / "reports" / "figures"

# A receita vencedora (01d–01f) treina em imagens RAW + augmentation jpeg+noise.
# Por padrão usamos RAW; um dataset pré-degradado (notebook 02) é uma abordagem
# ALTERNATIVA — se existir e você quiser usá-la, troque USE_RAW para False.
USE_RAW = True
processed_dirs = sorted(
    [d for d in (DATA_ROOT / "processed").glob("140k_*") if (d / "metadata.json").exists()],
    key=lambda d: d.stat().st_mtime, reverse=True
)
if processed_dirs and not USE_RAW:
    DATA_PATH = processed_dirs[0]
    meta = json.load(open(DATA_PATH / "metadata.json"))
    PREPROCESS_TAG = f"{meta['method']}_{meta['value']}"
else:
    DATA_PATH = RAW_DIR
    PREPROCESS_TAG = "raw"

MODEL_PATH = MODEL_DIR / f"efficientnet_b2_{PREPROCESS_TAG}.pth"

IMAGE_SIZE    = 260
BATCH_SIZE    = 32
NUM_WORKERS   = 4
NUM_EPOCHS    = 20
LEARNING_RATE = 1e-4   # confirmado pela busca de HP (01f): defaults já no bom regime; LR>3e-4 piora
PATIENCE      = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True   # input de tamanho fixo (260) -> autotuner acelera convoluções

for folder in [MODEL_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Dataset:", DATA_PATH)
print("Tag:", PREPROCESS_TAG)

## 1. Dados

EfficientNet-B2 espera imagens de 260x260 (resolução nativa do modelo).

In [ ]:
import sys
import random
from torch.utils.data import Subset

# aug_utils.py (ao lado do notebook) é módulo importável => RandomAugment pickla
# sob 'spawn' e num_workers>0 funciona no Windows.
sys.path.insert(0, str(Path.cwd()))
from aug_utils import RandomAugment

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
SAMPLE_FRACTION = 0.2  # 1.0 para usar tudo

# Augmentation vencedora das fases 01d–01f (jpeg+noise, p_apply=0.6).
# 01f confirmou que o ajuste de HP não superou os defaults; o ganho real veio
# DESTA augmentation (raw 0.632 -> jpeg+noise 0.687 na AUC cross-generator).
AUG_POOL   = ["jpeg", "noise"]
P_APPLY    = 0.6
AUG_RANGES = {"jpeg": (50, 90), "noise": (0.0, 0.05)}
AUGMENT    = RandomAugment(AUG_POOL, P_APPLY, AUG_RANGES)

train_transform = transforms.Compose([
    AUGMENT,                                   # jpeg+noise (treino apenas) — sobre a imagem PIL, antes do resize
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def sample_dataset(dataset, fraction, seed=42):
    n = int(len(dataset) * fraction)
    indices = random.Random(seed).sample(range(len(dataset)), n)
    return Subset(dataset, indices)

train_dataset = sample_dataset(datasets.ImageFolder(DATA_PATH / "train", transform=train_transform), SAMPLE_FRACTION)
valid_dataset = sample_dataset(datasets.ImageFolder(DATA_PATH / "valid", transform=eval_transform), SAMPLE_FRACTION)
test_dataset  = sample_dataset(datasets.ImageFolder(DATA_PATH / "test",  transform=eval_transform), SAMPLE_FRACTION)

# persistent_workers evita respawnar os workers a cada época (custo alto no Windows/spawn);
# pin_memory acelera a cópia host->GPU.
_loader_kw = dict(num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0, pin_memory=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  **_loader_kw)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, **_loader_kw)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, **_loader_kw)

CLASSES = train_dataset.dataset.classes
print("Classes:", CLASSES)
print(f"Augmentation treino: {AUG_POOL} (p_apply={P_APPLY}) ranges={AUG_RANGES}")
print(f"Treino:    {len(train_dataset):>6} | Validacao: {len(valid_dataset):>6} | Teste: {len(test_dataset):>6}")

## 2. Modelo

EfficientNet-B2 pré-treinada no ImageNet. O classificador final é substituído por dropout + camada linear binária.

In [ ]:
def build_efficientnet_b2(num_classes=2, dropout=0.3):
    model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, num_classes)
    )
    return model

model = build_efficientnet_b2().to(DEVICE)
print("Modelo carregado. Classificador final:", model.classifier)

## 3. Treinamento

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)

history = {"train_loss": [], "train_acc": [], "valid_loss": [], "valid_acc": [], "time": []}
best_val_loss = float("inf")
best_weights = None
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    start = time.time()

    model.train()
    train_loss, train_correct = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()

    model.eval()
    valid_loss, valid_correct = 0.0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            valid_loss += loss.item() * images.size(0)
            valid_correct += (outputs.argmax(1) == labels).sum().item()

    train_loss /= len(train_dataset)
    train_acc   = train_correct / len(train_dataset)
    valid_loss /= len(valid_dataset)
    valid_acc   = valid_correct / len(valid_dataset)
    elapsed     = time.time() - start

    history["train_loss"].append(round(train_loss, 6))
    history["train_acc"].append(round(train_acc, 6))
    history["valid_loss"].append(round(valid_loss, 6))
    history["valid_acc"].append(round(valid_acc, 6))
    history["time"].append(round(elapsed, 1))

    scheduler.step(valid_loss)

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {valid_loss:.4f} Acc: {valid_acc:.4f} | "
          f"{elapsed:.1f}s", flush=True)

    if valid_loss < best_val_loss:
        best_val_loss = valid_loss
        best_weights = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping na epoch {epoch+1}.", flush=True)
            break

model.load_state_dict(best_weights)
torch.save(best_weights, MODEL_PATH)

history_out = MODEL_DIR / f"efficientnet_b2_history_{PREPROCESS_TAG}.json"
with open(history_out, "w") as f:
    json.dump({"model": "efficientnet_b2", "preprocess": PREPROCESS_TAG, "epochs": [
        {"epoch": i+1, "train_loss": history["train_loss"][i], "train_acc": history["train_acc"][i],
         "val_loss": history["valid_loss"][i], "val_acc": history["valid_acc"][i], "time": history["time"][i]}
        for i in range(len(history["train_loss"]))
    ]}, f, indent=2)

print("Modelo salvo em:", MODEL_PATH)
print("Histórico salvo em:", history_out)

## 4. Curvas de Treinamento

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_range, history["train_loss"], label="Treino")
ax1.plot(epochs_range, history["valid_loss"], label="Validacao")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(epochs_range, history["train_acc"], label="Treino")
ax2.plot(epochs_range, history["valid_acc"], label="Validacao")
ax2.set_title("Acuracia")
ax2.set_xlabel("Epoch")
ax2.legend()

plt.suptitle(f"EfficientNet-B2 - Curvas de Treinamento ({PREPROCESS_TAG})")
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"efficientnet_b2_training_curves_{PREPROCESS_TAG}.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Avaliação no Teste

In [ ]:
model.eval()
all_labels, all_preds, all_probs = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        preds = outputs.argmax(1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

report = classification_report(all_labels, all_preds, target_names=CLASSES, output_dict=True)
auc = round(roc_auc_score(all_labels, all_probs), 6)

print(classification_report(all_labels, all_preds, target_names=CLASSES))
print("AUC-ROC:", auc)

metrics_out = MODEL_DIR / f"efficientnet_b2_metrics_{PREPROCESS_TAG}.json"
with open(metrics_out, "w") as f:
    json.dump({"model": "efficientnet_b2", "preprocess": PREPROCESS_TAG, "auc_roc": auc, "report": report}, f, indent=2)
print("Métricas salvas em:", metrics_out)

## 6. Matriz de Confusão e Curva ROC

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax1)
ax1.set_title(f"Matriz de Confusao - EfficientNet-B2 ({PREPROCESS_TAG})")
ax1.set_ylabel("Real")
ax1.set_xlabel("Predito")

fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc = roc_auc_score(all_labels, all_probs)
ax2.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
ax2.plot([0, 1], [0, 1], "--", color="gray")
ax2.set_title(f"Curva ROC - EfficientNet-B2 ({PREPROCESS_TAG})")
ax2.set_xlabel("FPR")
ax2.set_ylabel("TPR")
ax2.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / f"efficientnet_b2_confusion_roc_{PREPROCESS_TAG}.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Avaliação Cross-Generator (ArtiFact)

Teste de generalização real: o modelo é avaliado em faces de **outros geradores** (ArtiFact, sem StyleGAN). A mesma preprocess do treino é aplicada ao ArtiFact, e o conjunto é **balanceado** (mesmo nº de real/fake).

A célula **carrega o modelo do disco** (`MODEL_PATH`), então roda mesmo sem retreinar nesta sessão — basta ter executado as células de configuração (1, 2) e ter o `.pth` salvo.

In [ ]:
import io
import numpy as np
from PIL import Image, ImageFilter
from tqdm.notebook import tqdm
from aug_utils import artifact_split

ARTIFACT_DIR = DATA_ROOT / "raw" / "artifact_faces"
ARTIFACT_N_PER_CLASS = 5000   # por classe (balanceado)

# reconstrói a preprocess do treino para aplicar também ao ArtiFact (identidade se raw)
_pp = (None, None) if PREPROCESS_TAG == "raw" else (meta["method"], meta["value"])

def artifact_preprocess(img):
    method, value = _pp
    if method is None:
        return img
    if method == "jpeg":
        buf = io.BytesIO(); img.save(buf, format="JPEG", quality=int(value)); buf.seek(0)
        return Image.open(buf).copy()
    if method == "blur":
        return img.filter(ImageFilter.GaussianBlur(radius=float(value)))
    if method == "downscale":
        w, h = img.size
        return img.resize((max(1, int(w/value)), max(1, int(h/value))), Image.BILINEAR).resize((w, h), Image.BILINEAR)
    if method == "noise":
        arr = np.array(img, dtype=np.float32) + np.random.normal(0, float(value) * 255, np.array(img).shape)
        return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    if method == "median":
        return img.filter(ImageFilter.MedianFilter(size=int(value)))
    return img

class ArtifactDS(torch.utils.data.Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, l = self.items[i]
        return self.tf(artifact_preprocess(Image.open(p).convert("RGB"))), l

art_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def _list(folder):
    fs = []
    for e in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
        fs += list(folder.glob(e))
    return sorted(fs)

# METADE DE TESTE: disjunta da metade usada para selecionar augmentation/HP (01d/e/f).
# Assim o AUC cross-generator reportado NÃO é medido nos dados usados para escolher.
_real_pool = artifact_split(_list(ARTIFACT_DIR / "real"), which="test")
_fake_pool = artifact_split(_list(ARTIFACT_DIR / "fake"), which="test")
_rng = random.Random(42)
_real = _rng.sample(_real_pool, min(ARTIFACT_N_PER_CLASS, len(_real_pool)))
_fake = _rng.sample(_fake_pool, min(ARTIFACT_N_PER_CLASS, len(_fake_pool)))
art_items = [(p, 1) for p in _real] + [(p, 0) for p in _fake]   # fake=0, real=1 (igual ao 140k)
print(f"ArtiFact [metade de TESTE]: {len(_real)} real + {len(_fake)} fake | preprocess: {PREPROCESS_TAG}")

# carrega o modelo treinado do disco — roda mesmo sem ter retreinado nesta sessão
art_model = build_efficientnet_b2().to(DEVICE)
art_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
art_model.eval()

art_loader = DataLoader(ArtifactDS(art_items, art_tf), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)
y_true, y_prob = [], []
with torch.no_grad():
    for imgs, lbls in tqdm(art_loader, desc="ArtiFact"):
        imgs = imgs.to(DEVICE)
        prob = torch.softmax(art_model(imgs), dim=1)[:, 1].cpu().numpy()
        y_prob.extend(prob.tolist())
        y_true.extend(lbls.tolist())

y_true = np.array(y_true); y_prob = np.array(y_prob)
art_auc = roc_auc_score(y_true, y_prob)
art_preds = (y_prob >= 0.5).astype(int)

print(classification_report(y_true, art_preds, target_names=CLASSES))
print(f"AUC cross-generator (ArtiFact, metade de teste): {art_auc:.4f}")

art_metrics = {
    "model": "efficientnet_b2", "preprocess": PREPROCESS_TAG, "eval": "artifact_cross_generator",
    "artifact_split": "test", "auc_roc": round(float(art_auc), 6), "n_real": len(_real), "n_fake": len(_fake),
}
(MODEL_DIR / f"efficientnet_b2_artifact_{PREPROCESS_TAG}.json").write_text(json.dumps(art_metrics, indent=2))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
cm = confusion_matrix(y_true, art_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax1)
ax1.set_title(f"Confusão ArtiFact - EfficientNet-B2 ({PREPROCESS_TAG})")
ax1.set_ylabel("Real"); ax1.set_xlabel("Predito")
fpr, tpr, _ = roc_curve(y_true, y_prob)
ax2.plot(fpr, tpr, label=f"AUC = {art_auc:.4f}")
ax2.plot([0, 1], [0, 1], "--", color="gray")
ax2.set_title("ROC ArtiFact (cross-generator)")
ax2.set_xlabel("FPR"); ax2.set_ylabel("TPR"); ax2.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"efficientnet_b2_artifact_roc_{PREPROCESS_TAG}.png", dpi=150, bbox_inches="tight")
plt.show()